# Octahedron constellation around Base, with JWST as a reference

Run on Windows with your existing **LIFEpy / STK 13** environment and Astrogator available. This notebook follows the original `octahedron-constellation.py` in the same order, with a numbered explanation before each code cell.

- **JWST:** the reference satellite loaded from the supplied external ephemeris.
- **Base:** the reference spacecraft configured by the existing crossing-targeting helper, starting from the selected JWST separation epoch.
- **Octahedron:** six spacecraft with requested offsets of ±1,000 m along the three Base-centred L2 axes. The active configuration requests a **28-day station-keeping interval**, beginning **10 minutes after Base's separation epoch**.

## Before running

Place this notebook beside the original script, in the project directory containing `helpers`, and launch Jupyter with that directory as the **kernel working directory**. Keep your existing helper modules (`add_satellites.py`, `scenario_setup.py`, and `analysis.py`), `helpers/Maneuver-Summary-Detailed.rst`, and any other files those helpers require. Check the existing absolute JWST ephemeris path in **cell 9**. These dependencies are not embedded in this notebook.

Run **cells 1–18** in order to construct and run the scenario; **cells 19–23** load the report style, export reports, and parse CSV files. **Cell 5** is an optional, commented reconnection example, and **cell 24** retains the commented, deprecated cleaning code. Creating objects or coordinate systems again in the same scenario may cause duplicate-name errors; these cells retain the original script's behaviour. Report export does not save the STK scenario.

## What changed from the script

The two executable uses of `Path(__file__).resolve().parent` are replaced with `Path.cwd()` because Jupyter does not define `__file__`. All other executable code, configuration values, helper calls, and their order are preserved. The original cell markers become notebook boundaries; the final empty marker is omitted. The commented historical configurations and deprecated code are retained.

The explanations below use the linked official Ansys STK documentation for STK concepts and commands. The project-specific helper implementations were not supplied, so their detailed force models, targeting profiles, stopping conditions, tolerances, and export conventions must be checked in your existing `helpers` folder. Python syntax and notebook structure are checked; STK execution must be verified on your Windows installation.

## 1. User configuration

Define the constellation cases to create. Only `Octahedron_SK28d` is active; the other station-keeping intervals remain commented out exactly as in the script. Its six requested offsets are in metres: +X, −X, +Y, −Y, +Z, and −Z, each 1,000 m from Base. Opposite vertices are therefore 2,000 m apart; adjacent vertices are approximately 1,414 m apart at the requested geometry.

`station_keeping_interval_days` is passed to the position-hold helper; it is not a numerical integration step. `separation_delay_minutes` sets each constellation's release epoch relative to Base's separation. If enabling another configuration, keep its names and offset dictionary consistent. This cell only defines Python data; it does not create STK objects.

In [1]:
# Octahedron constellation configurations - list of all constellations to create
CONSTELLATION_CONFIGS = [
    # { # 0.5h, took approximately ? min to run
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK0p5h',
    #     'satellite_names': ['Octahedron_SK0p5h_Xp1000m', 'Octahedron_SK0p5h_Xm1000m', 'Octahedron_SK0p5h_Yp1000m', 'Octahedron_SK0p5h_Ym1000m', 'Octahedron_SK0p5h_Zp1000m', 'Octahedron_SK0p5h_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK0p5h_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},    # +X
    #         'Octahedron_SK0p5h_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},   # -X
    #         'Octahedron_SK0p5h_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},    # +Y
    #         'Octahedron_SK0p5h_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},   # -Y
    #         'Octahedron_SK0p5h_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},    # +Z
    #         'Octahedron_SK0p5h_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},   # -Z
    #     },
    #     'station_keeping_interval_days': 0.5 / 24, # 0.5 h in days
    #     'separation_delay_minutes': 10
    # },
    # { # 1h, took ? min to run
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK1h',
    #     'satellite_names': ['Octahedron_SK1h_Xp1000m', 'Octahedron_SK1h_Xm1000m', 'Octahedron_SK1h_Yp1000m', 'Octahedron_SK1h_Ym1000m', 'Octahedron_SK1h_Zp1000m', 'Octahedron_SK1h_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK1h_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},    # +X
    #         'Octahedron_SK1h_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},   # -X
    #         'Octahedron_SK1h_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},    # +Y
    #         'Octahedron_SK1h_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},   # -Y
    #         'Octahedron_SK1h_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},    # +Z
    #         'Octahedron_SK1h_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},   # -Z
    #     },
    #     'station_keeping_interval_days': 1 /24, # 1 h in days
    #     'separation_delay_minutes': 10
    # },
    # { # 3h, took ? min to run
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK3h',
    #     'satellite_names': ['Octahedron_SK3h_Xp1000m', 'Octahedron_SK3h_Xm1000m', 'Octahedron_SK3h_Yp1000m', 'Octahedron_SK3h_Ym1000m', 'Octahedron_SK3h_Zp1000m', 'Octahedron_SK3h_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK3h_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},    # +X
    #         'Octahedron_SK3h_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},   # -X
    #         'Octahedron_SK3h_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},    # +Y
    #         'Octahedron_SK3h_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},   # -Y
    #         'Octahedron_SK3h_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},    # +Z
    #         'Octahedron_SK3h_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},   # -Z
    #     },
    #     'station_keeping_interval_days': 3 /24, # 3 h in days
    #     'separation_delay_minutes': 10
    # },
    # { # 6h, took 36.5 min to run
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK6h',
    #     'satellite_names': ['Octahedron_SK6h_Xp1000m', 'Octahedron_SK6h_Xm1000m', 'Octahedron_SK6h_Yp1000m', 'Octahedron_SK6h_Ym1000m', 'Octahedron_SK6h_Zp1000m', 'Octahedron_SK6h_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK6h_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},    # +X
    #         'Octahedron_SK6h_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},   # -X
    #         'Octahedron_SK6h_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},    # +Y
    #         'Octahedron_SK6h_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},   # -Y
    #         'Octahedron_SK6h_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},    # +Z
    #         'Octahedron_SK6h_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},   # -Z
    #     },
    #     'station_keeping_interval_days': 6 /24, # 6 h in days
    #     'separation_delay_minutes': 10
    # },
    # { # 12h, took 16 min to run
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK12h',
    #     'satellite_names': ['Octahedron_SK12h_Xp1000m', 'Octahedron_SK12h_Xm1000m', 'Octahedron_SK12h_Yp1000m', 'Octahedron_SK12h_Ym1000m', 'Octahedron_SK12h_Zp1000m', 'Octahedron_SK12h_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK12h_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},    # +X
    #         'Octahedron_SK12h_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},   # -X
    #         'Octahedron_SK12h_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},    # +Y
    #         'Octahedron_SK12h_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},   # -Y
    #         'Octahedron_SK12h_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},    # +Z
    #         'Octahedron_SK12h_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},   # -Z
    #     },
    #     'station_keeping_interval_days': 12 /24, # 12 h in days
    #     'separation_delay_minutes': 10
    # },
    # {
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK1d',
    #     'satellite_names': ['Octahedron_SK1d_Xp1000m', 'Octahedron_SK1d_Xm1000m', 'Octahedron_SK1d_Yp1000m', 'Octahedron_SK1d_Ym1000m', 'Octahedron_SK1d_Zp1000m', 'Octahedron_SK1d_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK1d_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},    # +X
    #         'Octahedron_SK1d_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},   # -X
    #         'Octahedron_SK1d_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},    # +Y
    #         'Octahedron_SK1d_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},   # -Y
    #         'Octahedron_SK1d_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},    # +Z
    #         'Octahedron_SK1d_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},   # -Z
    #     },
    #     'station_keeping_interval_days': 1,
    #     'separation_delay_minutes': 10
    # },
    # {
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK3d',
    #     'satellite_names': ['Octahedron_SK3d_Xp1000m', 'Octahedron_SK3d_Xm1000m', 'Octahedron_SK3d_Yp1000m', 'Octahedron_SK3d_Ym1000m', 'Octahedron_SK3d_Zp1000m', 'Octahedron_SK3d_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK3d_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},
    #         'Octahedron_SK3d_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},
    #         'Octahedron_SK3d_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},
    #         'Octahedron_SK3d_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},
    #         'Octahedron_SK3d_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},
    #         'Octahedron_SK3d_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},
    #     },
    #     'station_keeping_interval_days': 3,
    #     'separation_delay_minutes': 10
    # },
    # {
    #     'num_satellites': 6,
    #     'base_name': 'Octahedron_SK7d',
    #     'satellite_names': ['Octahedron_SK7d_Xp1000m', 'Octahedron_SK7d_Xm1000m', 'Octahedron_SK7d_Yp1000m', 'Octahedron_SK7d_Ym1000m', 'Octahedron_SK7d_Zp1000m', 'Octahedron_SK7d_Zm1000m'],
    #     'offsets_m': {
    #         'Octahedron_SK7d_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},
    #         'Octahedron_SK7d_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},
    #         'Octahedron_SK7d_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},
    #         'Octahedron_SK7d_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},
    #         'Octahedron_SK7d_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},
    #         'Octahedron_SK7d_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},
    #     },
    #     'station_keeping_interval_days': 7,
    #     'separation_delay_minutes': 10
    # },
    {
        'num_satellites': 6,
        'base_name': 'Octahedron_SK28d',
        'satellite_names': ['Octahedron_SK28d_Xp1000m', 'Octahedron_SK28d_Xm1000m', 'Octahedron_SK28d_Yp1000m', 'Octahedron_SK28d_Ym1000m', 'Octahedron_SK28d_Zp1000m', 'Octahedron_SK28d_Zm1000m'],
        'offsets_m': {
            'Octahedron_SK28d_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},
            'Octahedron_SK28d_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},
            'Octahedron_SK28d_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},
            'Octahedron_SK28d_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},
            'Octahedron_SK28d_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},
            'Octahedron_SK28d_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},
        },
        'station_keeping_interval_days': 28,
        'separation_delay_minutes': 10
    },
    {
         'num_satellites': 6,
         'base_name': 'Octahedron_SK14d',
         'satellite_names': ['Octahedron_SK14d_Xp1000m', 'Octahedron_SK14d_Xm1000m', 'Octahedron_SK14d_Yp1000m', 'Octahedron_SK14d_Ym1000m', 'Octahedron_SK14d_Zp1000m', 'Octahedron_SK14d_Zm1000m'],
         'offsets_m': {
             'Octahedron_SK14d_Xp1000m': {'x': 1000.0, 'y': 0.0, 'z': 0.0},
             'Octahedron_SK14d_Xm1000m': {'x': -1000.0, 'y': 0.0, 'z': 0.0},
             'Octahedron_SK14d_Yp1000m': {'x': 0.0, 'y': 1000.0, 'z': 0.0},
             'Octahedron_SK14d_Ym1000m': {'x': 0.0, 'y': -1000.0, 'z': 0.0},
             'Octahedron_SK14d_Zp1000m': {'x': 0.0, 'y': 0.0, 'z': 1000.0},
             'Octahedron_SK14d_Zm1000m': {'x': 0.0, 'y': 0.0, 'z': -1000.0},
         },
         'station_keeping_interval_days': 14,
         'separation_delay_minutes': 10
     }
]


## 2. Imports

Import the original Python and STK 13 modules, then import the project's three helper modules. The `agi.stk13` packages provide STK Desktop automation, object types, utilities, colours, and Astrogator interfaces. The `helpers` package is project code, rather than part of the STK API.

Run this once after starting the kernel. If the STK import warning appears, resolve the environment problem before continuing: the original `try/except` prints a warning but does not stop execution. See the [STK Python Programmer's Guide](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm).

In [2]:
from pathlib import Path
import os
import importlib

try:
    from agi.stk13.stkdesktop import STKDesktop
    from agi.stk13.stkobjects import *
    from agi.stk13.stkutil import *
    from agi.stk13.utilities.colors import *
    from agi.stk13.stkobjects.astrogator import *
except:
    print("Failed to import stk modules...")

from helpers import add_satellites, scenario_setup, analysis


## 3. Reload helper modules and import functions

Reload the three helper modules and import the functions used later. This lets edits to the existing helper files take effect without restarting the notebook kernel; re-importing the functions binds their updated definitions.

On the first run, execute this after cell 2. Rerun it after editing a helper. Reloading does not rebuild satellites already created in STK. `set_constellation_attitudes` and `remove_position_spikes` are retained imports, but no active code below calls them.

In [3]:
import importlib
importlib.reload(scenario_setup)
importlib.reload(add_satellites)
importlib.reload(analysis)

# Re-import functions after reloading
from helpers.add_satellites import add_satellite_external_ephemeris, add_base_satellite_with_crossings, add_constellation_satellite_with_position_hold, set_constellation_attitudes
from helpers.scenario_setup import setup_l2_coordinate_system, create_cartesian_coordinate_components, set_current_time, time_string_to_epoch_seconds, epoch_seconds_to_time_string
from helpers.analysis import export_maneuver_summary_report, parse_maneuver_summary_report, export_relative_position_report, remove_position_spikes, parse_stk_position_txt


## 4. Start the STK application

Start a visible STK Desktop instance on Windows and obtain `stkRoot`, the root object used to create the scenario and send Connect commands. The original `userControl=False` is retained: STK remains under automation ownership rather than being configured to stay open after automation ends.

Run this once for a new STK session; rerunning starts another instance. This notebook retains the original Windows-only branch. See [starting and connecting to STK Desktop](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm).

In [4]:
if os.name == "nt":
    stk = STKDesktop.StartApplication(
        visible=True,
        userControl=False,
        )
    stkRoot = stk.Root
else:
    print("Only works on windows with the desktop application.")
    quit()


## 5. Optional: reconnect to an existing STK instance

This cell is intentionally commented out and does nothing during a normal run. After a kernel restart or crash, you can uncomment the two lines to attach to an STK instance instead of starting a new one in cell 4.

Attaching restores the application connection only; it does not recreate Python variables such as `scenario` and `all_constellations`. Recover those separately if continuing an existing session. See [STK Desktop attachment](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm).

In [5]:
# stk = STKDesktop.AttachToApplication()
# stkRoot = stk.Root


## 6. Create the scenario and set dates and units

Create `OctahedronConstellationScenario`, reset the Object Model units, and select Gregorian UTC (`UTCG`) dates and kilometre distances. Set the scenario interval to **26 December 2021–26 December 2025**, with **25 December 2021** as its epoch, and move the current animation time to the start.

The scenario interval is the analysis timeline; it does not itself propagate a satellite. The separation epochs and helper-defined mission sequences determine each simulated trajectory. Configuration offsets remain in metres because the helper arguments explicitly use `_m`.

In [6]:
print('Creating new scenario...')
stkRoot.NewScenario('OctahedronConstellationScenario')
scenario = stkRoot.CurrentScenario

# reset dimensions to default
dimensions = stkRoot.UnitPreferences
dimensions.ResetUnits()
dimensions.SetCurrentUnit("DateFormat", "UTCG")
dimensions.SetCurrentUnit("DistanceUnit", "km")

# Set scenario time bounds (JWST launch was 25 Dec 2021)
startTime = '26 Dec 2021 00:00:00.000'
stopTime = '26 Dec 2025 00:00:00.000'
epoch = '25 Dec 2021 00:00:00.000'
scenario.StartTime = startTime
scenario.StopTime = stopTime
scenario.Epoch = epoch
set_current_time(stkRoot, scenario, '26 Dec 2021 00:00:00.000')

print(f'Scenario created: {scenario.InstanceName}')


Creating new scenario...
Scenario created: OctahedronConstellationScenario


## 7. Set up the L2 coordinate system and graphics

Call the existing `setup_l2_coordinate_system` helper with diagnostic printing enabled. Later cells require the resulting system to be available as `CentralBody/Sun Syst_SEM_L2`.

In STK's Vector Geometry Tool (VGT), a coordinate system combines an origin point and axes. The exact L2 geometry and graphics settings created here are defined inside your helper; this one-line call does not expose them. See [VGT system construction](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VectorToolSystem.htm).

In [7]:
setup_l2_coordinate_system(stkRoot, verbose=True)


Setting up L2 coordinate system and graphics...
  L2 coordinate system created: Syst_SEM_L2
  Graphics setup: axes (cyan), XZ/YZ planes (gray), scale 10x
  Clipping distance set to 1e15 m


## 8. Create Cartesian calculation components in the L2 frame

Ask the helper to create Cartesian calculation components for `Syst_SEM_L2`, using `_L2` as the naming suffix. Astrogator calculation objects supply quantities such as position and velocity components for targeting results, reports, or stopping conditions.

This prepares quantities for later use; it does not apply a manoeuvre or execute a targeter. The helper determines the exact component names and settings. See [Astrogator calculation objects](https://help.agi.com/stk/Content/gator/ab-state.htm).

In [8]:
create_cartesian_coordinate_components(stkRoot, "CentralBody/Sun Syst_SEM_L2", "_L2", verbose=True)


Creating coordinate components with suffix "_L2" for system "CentralBody/Sun Syst_SEM_L2"
  Created component: Vx_L2
  Created component: Vy_L2
  Created component: Vz_L2
  Created component: X_L2
  Created component: Y_L2
  Created component: Z_L2
Coordinate components creation completed for system "CentralBody/Sun Syst_SEM_L2"


## 9. Load the JWST reference ephemeris

Pass the original absolute `.e` file path to `add_satellite_external_ephemeris` to create JWST. Change that path only if the file is stored elsewhere on your computer.

STK's External propagator reads supplied trajectory data and interpolates between its samples when needed. Loading an external ephemeris does not perform a new force-model integration of JWST. Ensure the file covers the times required by the helpers. See the [STK ephemeris file documentation](https://help.agi.com/stk/content/stk/importfiles-02.htm).

In [9]:
print('Adding satellite objects...')
JWSTephemerisFilePath = r'C:\Users\megap\Desktop\LIFE\Working\LIFE-Sem-Project\Previous-Project\horizon-to-STK\data\2025-11-06_horizons_JWST_hourly_from-launch.e'
JWST = add_satellite_external_ephemeris(scenario, 'JWST', JWSTephemerisFilePath)


Adding satellite objects...


## 10. Configure Base and its crossing-targeting sequence

Set Base's separation epoch to **6 October 2022 at 07:00 UTC** and call `add_base_satellite_with_crossings`, using JWST as the leader reference. Preserve the original arguments `num_crossings=4` and `min_propagate_duration=30`.

The crossing event definition, the units of the minimum-duration argument, the force model, and the chosen targeting profile are implemented in the helper and cannot be established from this script alone. Cell 12 explicitly runs Base's MCS. An Astrogator MCS is an ordered set of trajectory segments; see the [Astrogator introduction](https://help.agi.com/stk/Content/training/Gator_Circularize.htm).

In [10]:
base_separation_epoch = "6 Oct 2022 07:00:00.000 UTCG"

Base = add_base_satellite_with_crossings(
    scenario=scenario,
    stkRoot=stkRoot,
    satellite_name='Base',
    leader_satellite_name='JWST',
    separation_epoch=base_separation_epoch,
    num_crossings=4,
    min_propagate_duration=30,
    verbose=True
)


Creating base satellite: Base
  Number of crossings: 4
  Crossing interval: 30 days
Base satellite Base created successfully
  Leader: JWST
  Separation epoch: 6 Oct 2022 07:00:00.000 UTCG
  4-crossing orbit maintenance configured
  4 Differential Correctors set up for XZ plane crossings


## 11. Display the existing satellite tracks in the L2 frame

Add `Syst_SEM_L2` as a 3D orbit display system for all satellite objects currently present, and hide their default `InertialByWindow` orbit display. At this point the script has created JWST and Base; the constellation spacecraft are created later.

These commands affect displayed orbit tracks, not the propagation force model or spacecraft attitude. The frame must already exist in VGT. See [VO OrbitSystem](https://help.agi.com/stk/13.0.0/SubSystems/connectCmds/Content/cmd_VOOrbitSystem.htm).

In [11]:
stkRoot.ExecuteCommand(f'VO */Satellite/* OrbitSystem Add System "CentralBody/Sun Syst_SEM_L2" WindowID All')
stkRoot.ExecuteCommand(f'VO */Satellite/* OrbitSystem Modify System "InertialByWindow" WindowID All Show Off')


## 12. Run Base targeting, apply changes, and rerun nominally

Run Base's `Target_Sequence` with its active profiles enabled, then execute `ApplyAllProfileChanges` to transfer the active profile changes into the MCS. Clear the trial-trajectory graphics, switch the target sequence to nominal execution, and run the MCS again.

The nominal rerun uses the segment settings after those changes have been applied. The helper determines the profile type and its constraints. Check Base's targeting status in STK before proceeding. See [Target Sequence actions](https://help.agi.com/stk/Content/gator/mc-targeting.htm), [ApplyAllProfileChanges](https://help.agi.com/stk/Subsystems/connectCmds/Content/cxApplyAllProfileChanges.htm), and [ClearDWCGraphics](https://help.agi.com/stk/Subsystems/connectCmds/Content/cxClearDWCGraphics.htm).

In [12]:
print('Running Base satellite Mission Control Sequence...')
stkRoot.ExecuteCommand(f'Astrogator */Satellite/Base SetValue MainSequence.SegmentList.Target_Sequence.Action Run active profiles')
stkRoot.ExecuteCommand(f'Astrogator */Satellite/Base RunMCS')
stkRoot.ExecuteCommand(f'Astrogator */Satellite/Base ApplyAllProfileChanges')

# Clear Graphics, set Target Sequence to run nominal sequence and refresh graphics
stkRoot.ExecuteCommand(f'Astrogator */Satellite/Base ClearDWCGraphics')
stkRoot.ExecuteCommand(f'Astrogator */Satellite/Base SetValue MainSequence.SegmentList.Target_Sequence.Action Run nominal sequence')
stkRoot.ExecuteCommand(f'Astrogator */Satellite/Base RunMCS')


Running Base satellite Mission Control Sequence...


## 13. Create the Base-centred L2 coordinate system

Create `Syst_Base_L2` as an assembled VGT system: its origin is `Satellite/Base Center`, while its axes are `CentralBody/Sun SEM_L2`. This provides a moving origin at Base while retaining the selected L2 axis directions, so relative positions can be expressed in that frame.

The system is owned under `CentralBody/Sun`, but its origin is Base. This retains the original unconditional `Create` command; rerunning it after the system already exists can fail. See [assembled system syntax](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VectorToolSystem.htm).

In [13]:
stkRoot.ExecuteCommand(f'VectorTool * CentralBody/Sun Create System Syst_Base_L2 "Assembled" "Satellite/Base Center" "CentralBody/Sun SEM_L2"')


## 14. Create Cartesian calculation components relative to Base

Create another set of Cartesian calculation components, this time expressed in `Syst_Base_L2` and named with the `_Base` suffix. Position components in this system describe displacement from Base along its L2-aligned axes.

These components make relative quantities available to the project's subsequent targeting or reporting setup. Their creation alone does not enforce the requested offsets. See [Astrogator calculation objects](https://help.agi.com/stk/Content/gator/ab-state.htm).

In [14]:
create_cartesian_coordinate_components(stkRoot, "CentralBody/Sun Syst_Base_L2", "_Base", verbose=True)


Creating coordinate components with suffix "_Base" for system "CentralBody/Sun Syst_Base_L2"
  Created component: Vx_Base
  Created component: Vy_Base
  Created component: Vz_Base
  Created component: X_Base
  Created component: Y_Base
  Created component: Z_Base
Coordinate components creation completed for system "CentralBody/Sun Syst_Base_L2"


## 15. Create every configured octahedron

Loop through the active configurations. For each one, convert Base's separation epoch to seconds relative to the scenario epoch, add the requested delay in seconds, and convert back to a date string. With the current configuration, release is **6 October 2022 at 07:10 UTC**.

For each satellite, pass its name, Base leader, release epoch, three metre offsets, and station-keeping interval to `add_constellation_satellite_with_position_hold`. Store the returned objects and configuration in `all_constellations` for later use. The helper configures the position-hold implementation; its name does not establish continuous thrust or exact offset maintenance between corrections. The next cell explicitly runs the configured MCSs.

In [15]:
# Store all constellation satellites
all_constellations = {}

# Iterate through each configuration
for config_idx, config in enumerate(CONSTELLATION_CONFIGS):
    print(f'\n{"="*80}')
    print(f'Creating constellation {config_idx+1}/{len(CONSTELLATION_CONFIGS)}: {config["base_name"]}')
    print(f'{"="*80}')
    
    # Calculate separation epoch for this constellation
    constellation_separation_epoch = epoch_seconds_to_time_string(
        scenario, 
        time_string_to_epoch_seconds(scenario, base_separation_epoch) + 
        config['separation_delay_minutes'] * 60
    )
    print(f'Constellation separation epoch: {constellation_separation_epoch}')
    print(f'Number of satellites: {config["num_satellites"]}')
    print(f'Station-keeping interval: {config["station_keeping_interval_days"]} days')
    
    # Store satellites for this constellation
    constellation_satellites = {}
    
    # Create each satellite in the constellation
    for i, sat_name in enumerate(config['satellite_names']):
        print(f'\nCreating satellite {i+1}/{config["num_satellites"]}: {sat_name}')
        
        # Get offsets for this satellite
        offsets = config['offsets_m'][sat_name]
        
        # Create constellation satellite
        satellite = add_constellation_satellite_with_position_hold(
            scenario=scenario,
            stkRoot=stkRoot,
            satellite_name=sat_name,
            leader_satellite_name='Base',
            separation_epoch=constellation_separation_epoch,
            x_offset_m=offsets['x'],
            y_offset_m=offsets['y'],
            z_offset_m=offsets['z'],
            station_keeping_interval_days=config['station_keeping_interval_days'],
            verbose=True
        )
        
        constellation_satellites[sat_name] = satellite
    
    # Store this constellation
    all_constellations[config['base_name']] = {
        'config': config,
        'satellites': constellation_satellites,
        'separation_epoch': constellation_separation_epoch
    }



Creating constellation 1/2: Octahedron_SK28d
Constellation separation epoch: 06 Oct 2022 07:10:00.000 UTCG
Number of satellites: 6
Station-keeping interval: 28 days

Creating satellite 1/6: Octahedron_SK28d_Xp1000m
Creating constellation satellite: Octahedron_SK28d_Xp1000m
Constellation satellite Octahedron_SK28d_Xp1000m created successfully
  Leader: Base
  Separation epoch: 06 Oct 2022 07:10:00.000 UTCG
  Target offset: (1000.0, 0.0, 0.0) m
  Station-keeping interval: 28 days

Creating satellite 2/6: Octahedron_SK28d_Xm1000m
Creating constellation satellite: Octahedron_SK28d_Xm1000m
Constellation satellite Octahedron_SK28d_Xm1000m created successfully
  Leader: Base
  Separation epoch: 06 Oct 2022 07:10:00.000 UTCG
  Target offset: (-1000.0, 0.0, 0.0) m
  Station-keeping interval: 28 days

Creating satellite 3/6: Octahedron_SK28d_Yp1000m
Creating constellation satellite: Octahedron_SK28d_Yp1000m
Constellation satellite Octahedron_SK28d_Yp1000m created successfully
  Leader: Base
  S

## 16. Run the constellation mission control sequences

For each formation spacecraft, enable active profiles in the nested `Sequence/Target_Sequence`, run its MCS, then switch that target sequence's action to nominal execution. The code catches exceptions so one satellite's failure does not stop the remaining runs.

Unlike Base's block, this cell has no explicit `ApplyAllProfileChanges` call or second nominal MCS run. Its printed completion message means no exception was caught; inspect STK's profile status and results to establish convergence and achieved formation accuracy. See [Target Sequence configuration](https://help.agi.com/stk/Content/gator/mc-targeting.htm).

In [16]:
print(f'\n{"="*80}')
print(f'RUNNING MISSION CONTROL SEQUENCES')
print(f'{"="*80}')

for config in CONSTELLATION_CONFIGS:
    print(f'\n--- Running MCS for {config["base_name"]} ---')
    
    for sat_name in config['satellite_names']:
        print(f'Running MCS for {sat_name}...')
        try:
            stkRoot.ExecuteCommand(f'Astrogator */Satellite/{sat_name} SetValue MainSequence.SegmentList.Sequence.SegmentList.Target_Sequence.Action Run active profiles')
            stkRoot.ExecuteCommand(f'Astrogator */Satellite/{sat_name} RunMCS')
            stkRoot.ExecuteCommand(f'Astrogator */Satellite/{sat_name} SetValue MainSequence.SegmentList.Sequence.SegmentList.Target_Sequence.Action Run nominal sequence')
            print(f'  ✓ {sat_name} MCS completed successfully')
        except Exception as e:
            print(f'  ✗ {sat_name} MCS failed: {str(e)}')



RUNNING MISSION CONTROL SEQUENCES

--- Running MCS for Octahedron_SK28d ---
Running MCS for Octahedron_SK28d_Xp1000m...
  ✓ Octahedron_SK28d_Xp1000m MCS completed successfully
Running MCS for Octahedron_SK28d_Xm1000m...
  ✓ Octahedron_SK28d_Xm1000m MCS completed successfully
Running MCS for Octahedron_SK28d_Yp1000m...
  ✓ Octahedron_SK28d_Yp1000m MCS completed successfully
Running MCS for Octahedron_SK28d_Ym1000m...
  ✓ Octahedron_SK28d_Ym1000m MCS completed successfully
Running MCS for Octahedron_SK28d_Zp1000m...
  ✓ Octahedron_SK28d_Zp1000m MCS completed successfully
Running MCS for Octahedron_SK28d_Zm1000m...
  ✓ Octahedron_SK28d_Zm1000m MCS completed successfully

--- Running MCS for Octahedron_SK14d ---
Running MCS for Octahedron_SK14d_Xp1000m...
  ✓ Octahedron_SK14d_Xp1000m MCS completed successfully
Running MCS for Octahedron_SK14d_Xm1000m...
  ✓ Octahedron_SK14d_Xm1000m MCS completed successfully
Running MCS for Octahedron_SK14d_Yp1000m...
  ✓ Octahedron_SK14d_Yp1000m MCS comp

## 17. Move the animation time to the first release

Rewind the scenario and set its current time to the separation epoch stored for the first constellation. With the active configuration, this selects **6 October 2022 at 07:10 UTC**, making the release geometry convenient to inspect.

Changing animation time does not rerun targeting or propagation. This cell requires `all_constellations` to contain at least one configuration from cell 15.

In [17]:
stkRoot.Rewind()
# Use the first constellation's separation epoch
first_separation_epoch = list(all_constellations.values())[0]['separation_epoch']
print(f"\nSetting current time to first constellation release: {first_separation_epoch}")
set_current_time(stkRoot, scenario, first_separation_epoch)



Setting current time to first constellation release: 06 Oct 2022 07:10:00.000 UTCG


## 18. Print the configuration summary

Print the leader and Base names, release epochs, station-keeping intervals, satellite names, and requested offsets for every configured octahedron. The total includes Base and the six spacecraft per active configuration, but excludes the JWST reference satellite; the current total is seven.

This is a summary of configuration inputs, rather than a measurement of propagated positions or a convergence check. Use the exported position reports and STK targeting results to assess the achieved formation.

In [18]:
print(f'\n{"="*80}')
print(f'COMPLETE CONSTELLATION SUMMARY')
print(f'{"="*80}')
print(f'Base satellite: Base')
print(f'Leader satellite: JWST')
print(f'Base separation: {base_separation_epoch}')

for constellation_name, constellation_data in all_constellations.items():
    config = constellation_data['config']
    separation_epoch = constellation_data['separation_epoch']
    
    print(f'\n--- {constellation_name} ({config["station_keeping_interval_days"]}-day station-keeping) ---')
    print(f'Constellation separation: {separation_epoch}')
    print(f'Number of satellites: {config["num_satellites"]}')
    print(f'Station-keeping interval: {config["station_keeping_interval_days"]} days')
    print(f'Satellite configuration:')
    for sat_name in config['satellite_names']:
        offsets = config['offsets_m'][sat_name]
        print(f'  {sat_name}: X={offsets["x"]:+7.1f}m, Y={offsets["y"]:+7.1f}m, Z={offsets["z"]:+7.1f}m')

total_satellites = 1 + sum(config['num_satellites'] for config in CONSTELLATION_CONFIGS)
print(f'\nTotal constellation satellites: {total_satellites} (1 Base + {len(CONSTELLATION_CONFIGS)} octahedrons × 6 satellites)')
print(f'Station-keeping intervals: {", ".join(str(c["station_keeping_interval_days"]) for c in CONSTELLATION_CONFIGS)} days')
print(f'\nConstellation setup complete! 🚀')



COMPLETE CONSTELLATION SUMMARY
Base satellite: Base
Leader satellite: JWST
Base separation: 6 Oct 2022 07:00:00.000 UTCG

--- Octahedron_SK28d (28-day station-keeping) ---
Constellation separation: 06 Oct 2022 07:10:00.000 UTCG
Number of satellites: 6
Station-keeping interval: 28 days
Satellite configuration:
  Octahedron_SK28d_Xp1000m: X=+1000.0m, Y=   +0.0m, Z=   +0.0m
  Octahedron_SK28d_Xm1000m: X=-1000.0m, Y=   +0.0m, Z=   +0.0m
  Octahedron_SK28d_Yp1000m: X=   +0.0m, Y=+1000.0m, Z=   +0.0m
  Octahedron_SK28d_Ym1000m: X=   +0.0m, Y=-1000.0m, Z=   +0.0m
  Octahedron_SK28d_Zp1000m: X=   +0.0m, Y=   +0.0m, Z=+1000.0m
  Octahedron_SK28d_Zm1000m: X=   +0.0m, Y=   +0.0m, Z=-1000.0m

--- Octahedron_SK14d (14-day station-keeping) ---
Constellation separation: 06 Oct 2022 07:10:00.000 UTCG
Number of satellites: 6
Station-keeping interval: 14 days
Satellite configuration:
  Octahedron_SK14d_Xp1000m: X=+1000.0m, Y=   +0.0m, Z=   +0.0m
  Octahedron_SK14d_Xm1000m: X=-1000.0m, Y=   +0.0m, Z=   

## 19. Load the detailed manoeuvre report style

Load `helpers/Maneuver-Summary-Detailed.rst` from the kernel working directory. STK's `ReportStyle` command copies the style into the user's local style configuration so reports can use it; an existing writable copy with the same name can be overwritten.

This is the first notebook-specific path change: `Path.cwd()` replaces the script's `Path(__file__).resolve().parent`. Keep the working directory at the original script's directory. See [ReportStyle](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_ReportStyle.htm).

In [19]:
# Load the detailed maneuver summary report style
# located in the helpers directory
stkRoot.ExecuteCommand(f'ReportStyle * Load "{str((Path.cwd() / "helpers" / "Maneuver-Summary-Detailed.rst").absolute())}"')


## 20. Export manoeuvre summary text reports

Call `export_maneuver_summary_report` for every formation spacecraft and store the returned file paths in `exported_files`. Base and JWST are not included in these loops.

The helper controls the report-generation command, chosen style, output path, and exported columns. This step exports the computed manoeuvre information; it does not add or execute burns. STK report contents depend on their styles and data providers; see [generating reports](https://help.agi.com/stk/Content/stk/report.htm).

In [20]:
exported_files = []

for config in CONSTELLATION_CONFIGS:
    for sat_name in config['satellite_names']:
        txt_file = export_maneuver_summary_report(stkRoot, sat_name, verbose=False)
        exported_files.append(txt_file)


## 21. Convert manoeuvre summaries to CSV

Parse each text report recorded in `exported_files` using the existing analysis helper. `delete_source=False` retains the original text reports, while `verbose=False` suppresses routine helper messages.

Parsing errors are printed and the loop continues with the next file. The parser defines the CSV format and units; keep those conventions when analysing manoeuvre magnitudes. This is Python post-processing of exported STK data, rather than another propagation.

In [21]:
for txt_file in exported_files:
    try:
        parse_maneuver_summary_report(txt_file, delete_source=False, verbose=False)
    except Exception as e:
        print(f'✗ Failed to parse {txt_file}: {str(e)}')


## 22. Export relative positions every ten minutes

Export relative-position reports for each formation spacecraft with a **600-second reporting step**. This controls report sampling, not the propagator's integration step. For the current release at 07:10 UTC and ten-minute delay, the preserved calculation begins reporting at **08:00:01 UTC** on 6 October 2022.

The original comments describe a one-second shift to avoid anomalous samples previously observed at manoeuvre epochs. That is a project-specific workaround, not a general STK guarantee. The helper defines the report frame, columns, units, and end time.

The original loop uses `constellation_separation_epoch` left from the final creation-loop configuration. This works for the current single case (or common release times); configurations with different release delays would need their own stored epoch. That existing behaviour is preserved here.

In [22]:
# Export for all constellation satellites
for config in CONSTELLATION_CONFIGS:
    for sat_name in config['satellite_names']:
        # Calculate start time for report (next full hour after separation) - WITHOUT UTCG for this connect command
        start_epoch = time_string_to_epoch_seconds(scenario, constellation_separation_epoch) + ((-config['separation_delay_minutes']) % 60) * 60
        # The next step is very important  to ensure clean data.
        # If a datapoint is exported exactly at the time of a maneuver, STK outputs a value which is off by several orders of magnitude.
        # To avoid this, the export start time is set to one second after the full hour.
        # With the regular time step of 10 minutes, this ensures no data point coincides with a maneuver time (set to be in regular multiples or fractions of hours).
        start_epoch += 1  # move to next full hour
        start_time_str = epoch_seconds_to_time_string(scenario, start_epoch, append_UTCG=False)
        
        export_relative_position_report(stkRoot, sat_name, 
                        time_period_start=start_time_str,
                        time_step_seconds=600,  # 10 minutes
                        verbose=False)


## 23. Convert relative-position reports to CSV

Read each expected text report from `data-export/temp/position` under the kernel working directory and pass it to `parse_stk_position_txt`. `trim_data=False` retains all data accepted by the parser, consistent with the original comment that trimming is handled during export. Errors are printed without stopping later files.

This is the second notebook-specific path change: `Path.cwd()` replaces the script directory expression. The filenames and directory layout remain the same. The commented alternative parser is preserved and is not executed.

In [23]:
# Parse without trimming (all data), since this functionality has since been integrated into the export function
for config in CONSTELLATION_CONFIGS:
    for sat_name in config['satellite_names']:
        txt_file = str(Path.cwd() / 'data-export' / 'temp' / 'position' / f'{sat_name}.txt')
        try:
            parse_stk_position_txt(txt_file, trim_data=False, verbose=False)
            # parse_relative_position_report(txt_file, trim_data=False, verbose=False)
        except Exception as e:
            print(f'✗ Failed to parse {txt_file}: {str(e)}')
            


## 24. Deprecated: position-spike removal, retained as comments

This entire cell is commented out and has no effect during normal execution. It preserves the former cleaning step, which the original script marks as deprecated after introducing the one-second reporting shift in cell 22.

The comments describe replacing isolated outliers with neighbouring values. Do not treat these comments as an active processing step. Their historical `__file__` path is left untouched because the block is inactive; adapting and enabling that old workflow would be a separate code change.

In [24]:
# DEPRECATED: Issue solved by shifting export start time by 1 second to avoid maneuvers
# 
# Remove spikes from all constellation satellites position data
# This was added after observing spikes (single data values which are off by orders of magnitude)
# in the relative position data
# Replaces the spikes with averaged values from neighboring data points
# print(f'\n{"="*80}')
# print('REMOVING POSITION DATA SPIKES')
# print(f'{"="*80}')

# for config in CONSTELLATION_CONFIGS:
#     for sat_name in config['satellite_names']:
#         csv_file = str(Path(__file__).resolve().parent / 'data-export' / 'positions' / f'{sat_name}.csv')
#         try:
#             remove_position_spikes(csv_file,
#                                    window_size=5,
#                                    threshold_mad=5, # empirical, set by trying out and looking at the data graphs
#                                    verbose=True
#                                    )
#         except Exception as e:
#             print(f'✗ Failed to clean spikes in {sat_name}: {str(e)}')

# print(f'\n{"="*80}')
# print('SPIKE REMOVAL COMPLETE')
# print(f'{"="*80}')


## Official Ansys STK documentation consulted

The links beside the relevant cells provide the source for the STK concepts used in their explanations. These are the STK 13 help pages and the `agi.stk13` Python guide matching the retained API.

- [STK Python Programmer's Guide](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm): STK module organisation, application start/attach, root access, and automation ownership.
- [Astrogator introduction](https://help.agi.com/stk/Content/training/Gator_Circularize.htm): mission control sequences and trajectory computation.
- [Target Sequence](https://help.agi.com/stk/Content/gator/mc-targeting.htm): active profiles, nominal execution, and targeting status.
- [ApplyAllProfileChanges](https://help.agi.com/stk/Subsystems/connectCmds/Content/cxApplyAllProfileChanges.htm) and [ClearDWCGraphics](https://help.agi.com/stk/Subsystems/connectCmds/Content/cxClearDWCGraphics.htm): applying profile changes and clearing calculation graphics.
- [VectorTool System](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_VectorToolSystem.htm): assembled coordinate systems made from a point and reference axes.
- [Calculation Object Components](https://help.agi.com/stk/Content/gator/ab-state.htm): Cartesian quantities for targeting and analysis.
- [VO OrbitSystem](https://help.agi.com/stk/13.0.0/SubSystems/connectCmds/Content/cmd_VOOrbitSystem.htm): selecting the frame for displayed orbit tracks.
- [Ephemeris File Format](https://help.agi.com/stk/content/stk/importfiles-02.htm): external trajectory data and interpolation.
- [ReportStyle](https://help.agi.com/stk/Subsystems/connectCmds/Content/cmd_ReportStyle.htm) and [Generating Reports and Graphs](https://help.agi.com/stk/Content/stk/report.htm): report styles and report generation.

The configuration values, function arguments, loop behaviour, and project-specific reporting workaround are explained from the supplied script. The linked documentation does not establish the unseen helper implementations.